# 14. Topic Group 생성 및 검증

카테고리/긍부정 그룹별 세부 topic 10-15개를 GPT-5-5로 3-5개 상위 topic_group으로 묶습니다.

- `기타`, `미분류`, `전반적 긍정/부정`, `LLM_FALLBACK_REQUIRED`는 모두 `기타` 그룹으로 통합합니다.
- 세부 topic은 topic명과 description을 함께 보고 의미가 가까운 것끼리 묶습니다.
- 이 노트북에서는 `topic_group` 매핑 테이블만 생성/검증합니다.
- 원본 row에 topic_group까지 붙이는 Tableau 최종 산출물은 15번 노트북에서 생성합니다.


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import taxonomy.topic_group_generator as topic_group_generator

importlib.reload(config_loader)
importlib.reload(topic_group_generator)

from common.config_loader import load_config, get_output_table, get_reference_table
from taxonomy.topic_group_generator import (
    generate_and_save_topic_groups,
)

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

print("topic_pool =", get_output_table(config, "topic_pool"))
print("topic_group =", get_output_table(config, "topic_group"))


In [ ]:
# 실행 옵션
# 최초 검증 시에는 LIMIT_GROUPS=2~3으로 돌리고 결과가 좋으면 None으로 전체 실행하세요.
LIMIT_GROUPS = 3
MODEL_KEY = "gpt_55"

result = generate_and_save_topic_groups(
    spark,
    config,
    limit_groups=LIMIT_GROUPS,
    model_key=MODEL_KEY,
    write_mode="replace_groups",
)

result


In [ ]:
# Topic -> Topic Group 매핑 확인
topic_group_table = get_output_table(config, "topic_group")
category_mapping_table = get_reference_table(config, "category_mapping_table")

topic_group_df = spark.table(topic_group_table).where(F.col("prompt_version") == config["version"]["prompt_version"])

display(
    topic_group_df.alias("g")
    .join(
        spark.table(category_mapping_table).alias("m"),
        on=["cate_1_depth", "cate_2_depth"],
        how="left",
    )
    .select(
        "g.cate_1_depth",
        "m.cate_1_depth_kor",
        "g.cate_2_depth",
        "m.cate_2_depth_kor",
        "g.sc_measurement",
        "g.topic_group_order",
        "g.topic_group",
        "g.topic",
        "g.topic_description",
        "g.grouping_reason",
        "g.is_special_group",
    )
    .orderBy("g.cate_1_depth", "g.cate_2_depth", "g.sc_measurement", "g.topic_group_order", "g.topic_order")
)
